In [ ]:
import mysql.connector
import pandas as pd
from tqdm import tqdm

CFG = {
    "user": "root",
    "password": "1234",
    "host": "localhost",
    "database": "flights_db"
}

SCHEMA_FILE = "01_schema.sql"
CSV_FILE = "Dataset_OData_Final_Martin_Otero.csv"
CHUNKSIZE = 100_000

# Columnas esperadas del CSV (según tu lista)
CSV_COLS = [
    'FlightDate','sched_dep_dt','sched_arr_dt','dep_dt','arr_dt',
    'Airline','Origin','Dest','Cancelled','Diverted',
    'DepDelayMinutes','DepDelay','ArrDelayMinutes','AirTime',
    'CRSElapsedTime','ActualElapsedTime','Distance',
    'Marketing_Airline_Network','Operated_or_Branded_Code_Share_Partners',
    'DOT_ID_Marketing_Airline','IATA_Code_Marketing_Airline','Flight_Number_Marketing_Airline',
    'Operating_Airline','DOT_ID_Operating_Airline','IATA_Code_Operating_Airline',
    'Tail_Number','Flight_Number_Operating_Airline',
    'OriginAirportID','OriginAirportSeqID','OriginCityMarketID','OriginCityName',
    'OriginState','OriginStateFips','OriginStateName','OriginWac',
    'DestAirportID','DestAirportSeqID','DestCityMarketID','DestCityName',
    'DestState','DestStateFips','DestStateName','DestWac',
    'DepDel15','DepartureDelayGroups','DepTimeBlk','TaxiOut','WheelsOff','WheelsOn','TaxiIn',
    'ArrDelay','ArrDel15','ArrivalDelayGroups','ArrTimeBlk',
    'DistanceGroup','DivAirportLandings',
    'origin_lat','origin_lon','dest_lat','dest_lon',
    'MODEL','aircraft_age'
]

def exec_sql_file(cursor, path):
    with open(path, "r", encoding="utf-8") as f:
        sql = f.read()
    for statement in sql.split(";"):
        stmt = statement.strip()
        if stmt:
            cursor.execute(stmt)

def upsert_airline(cur, iata_code, name=None, dot_id=None):
    if iata_code is None:
        return None
    cur.execute("SELECT airline_id FROM airline WHERE iata_code=%s", (iata_code,))
    row = cur.fetchone()
    if row: return row[0]
    cur.execute("INSERT INTO airline (iata_code, name, dot_id) VALUES (%s,%s,%s)", (iata_code, name, dot_id))
    return cur.lastrowid

def upsert_airport(cur, iata, name=None, city=None, state=None, country=None, lat=None, lon=None,
                   airport_seq_id=None, city_market_id=None, state_fips=None, wac=None):
    if iata is None:
        return None
    cur.execute("SELECT airport_id FROM airport WHERE iata=%s", (iata,))
    row = cur.fetchone()
    if row: return row[0]
    cur.execute("""
        INSERT INTO airport (iata, name, city, state, country, lat, lon, airport_seq_id, city_market_id, state_fips, wac)
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, (iata, name, city, state, country, lat, lon, airport_seq_id, city_market_id, state_fips, wac))
    return cur.lastrowid

def upsert_aircraft_model(cur, model_code):
    if model_code is None:
        return None
    cur.execute("SELECT model_id FROM aircraft_model WHERE model_code=%s", (model_code,))
    row = cur.fetchone()
    if row: return row[0]
    cur.execute("INSERT INTO aircraft_model (model_code) VALUES (%s)", (model_code,))
    return cur.lastrowid

def upsert_aircraft(cur, tail_number, model_code=None, aircraft_age=None):
    if tail_number is None:
        return None
    cur.execute("SELECT aircraft_id FROM aircraft WHERE tail_number=%s", (tail_number,))
    row = cur.fetchone()
    if row: return row[0]
    model_id = upsert_aircraft_model(cur, model_code) if model_code else None
    cur.execute("INSERT INTO aircraft (tail_number, model_id, aircraft_age) VALUES (%s,%s,%s)",
                (tail_number, model_id, aircraft_age))
    return cur.lastrowid

def upsert_performance(cur, dep_groups, dep_blk, arr_groups, arr_blk, wheels_off, wheels_on):
    # Si todos son None, no crea registro
    if not any([dep_groups, dep_blk, arr_groups, arr_blk, wheels_off, wheels_on]):
        return None
    cur.execute("""
        SELECT perf_id FROM flight_performance
        WHERE departure_delay_groups <=> %s AND dep_time_blk <=> %s
          AND arrival_delay_groups <=> %s AND arr_time_blk <=> %s
          AND wheels_off <=> %s AND wheels_on <=> %s
    """, (dep_groups, dep_blk, arr_groups, arr_blk, wheels_off, wheels_on))
    row = cur.fetchone()
    if row: return row[0]
    cur.execute("""
        INSERT INTO flight_performance (departure_delay_groups, dep_time_blk, arrival_delay_groups, arr_time_blk, wheels_off, wheels_on)
        VALUES (%s,%s,%s,%s,%s,%s)
    """, (dep_groups, dep_blk, arr_groups, arr_blk, wheels_off, wheels_on))
    return cur.lastrowid

def main():
    conn = mysql.connector.connect(**CFG)
    cur = conn.cursor()
    print("Conectado.")

    # 1) Crear esquema
    exec_sql_file(cur, SCHEMA_FILE)
    conn.commit()
    print("Esquema creado.")

    # 2) Cargar CSV en chunks e insertar en flight con FKs
    total_inserted = 0
    for chunk in tqdm(pd.read_csv(CSV_FILE, dtype=str, chunksize=CHUNKSIZE), desc="Procesando CSV"):
        chunk = chunk.where(pd.notnull(chunk), None)
        # Validar columnas mínimas
        missing = [c for c in CSV_COLS if c not in chunk.columns]
        if missing:
            raise ValueError(f"Faltan columnas en el CSV: {missing}")

        batch_values = []
        for _, r in chunk.iterrows():
            # Upsert dimensiones
            airline_id = upsert_airline(cur, r['Airline'], name=None, dot_id=r['DOT_ID_Marketing_Airline'])
            origin_id = upsert_airport(cur, r['Origin'], lat=r['origin_lat'], lon=r['origin_lon'],
                                       airport_seq_id=r['OriginAirportSeqID'], city_market_id=r['OriginCityMarketID'],
                                       state_fips=r['OriginStateFips'], wac=r['OriginWac'], name=r['OriginCityName'],
                                       city=r['OriginCityName'], state=r['OriginState'], country=None)
            dest_id = upsert_airport(cur, r['Dest'], lat=r['dest_lat'], lon=r['dest_lon'],
                                     airport_seq_id=r['DestAirportSeqID'], city_market_id=r['DestCityMarketID'],
                                     state_fips=r['DestStateFips'], wac=r['DestWac'], name=r['DestCityName'],
                                     city=r['DestCityName'], state=r['DestState'], country=None)
            aircraft_id = upsert_aircraft(cur, r['Tail_Number'], model_code=r['MODEL'], aircraft_age=r['aircraft_age'])
            perf_id = upsert_performance(cur,
                                         r['DepartureDelayGroups'], r['DepTimeBlk'],
                                         r['ArrivalDelayGroups'], r['ArrTimeBlk'],
                                         r['WheelsOff'], r['WheelsOn'])

            # Valores para tabla flight
            values = (
                r['FlightDate'], r['sched_dep_dt'], r['sched_arr_dt'], r['dep_dt'], r['arr_dt'],
                airline_id, aircraft_id, origin_id, dest_id, perf_id,
                r['Cancelled'] or 0, r['Diverted'] or 0, r['DepDelayMinutes'], r['DepDelay'],
                r['ArrDelayMinutes'], r['ArrDelay'], r['AirTime'], r['CRSElapsedTime'],
                r['ActualElapsedTime'], r['Distance'], r['Marketing_Airline_Network'],
                r['Operated_or_Branded_Code_Share_Partners'], r['DOT_ID_Marketing_Airline'],
                r['IATA_Code_Marketing_Airline'], r['Flight_Number_Marketing_Airline'],
                r['Operating_Airline'], r['DOT_ID_Operating_Airline'], r['IATA_Code_Operating_Airline'],
                r['Flight_Number_Operating_Airline'], r['DepDel15'] or 0, r['ArrDel15'] or 0,
                r['DistanceGroup'], r['DivAirportLandings'],
                'faa' if r['MODEL'] else 'inferred'
            )
            batch_values.append(values)

        insert_sql = """
        INSERT INTO flight (
          flight_date, sched_dep_dt, sched_arr_dt, dep_dt, arr_dt,
          airline_id, aircraft_id, origin_id, dest_id, perf_id,
          cancelled, diverted, dep_delay_minutes, dep_delay,
          arr_delay_minutes, arr_delay, airtime, crs_elapsed_time,
          actual_elapsed_time, distance, marketing_airline_network,
          operated_or_branded_code_share_partners, dot_id_marketing_airline,
          iata_code_marketing_airline, flight_number_marketing_airline,
          operating_airline, dot_id_operating_airline, iata_code_operating_airline,
          flight_number_operating_airline, dep_del15, arr_del15,
          distance_group, div_airport_landings, model_source
        )
        VALUES (
          %s,%s,%s,%s,%s,
          %s,%s,%s,%s,%s,
          %s,%s,%s,%s,
          %s,%s,%s,%s,
          %s,%s,%s,
          %s,%s,%s,
          %s,%s,%s,
          %s,%s,%s,%s,
          %s,%s,%s
        )
        """
        if batch_values:
            cur.executemany(insert_sql, batch_values)
            conn.commit()
            total_inserted += len(batch_values)

    print(f"Filas insertadas en flight: {total_inserted}")

    cur.close()
    conn.close()
    print("Proceso completado.")

if __name__ == "__main__":
    main()


Insertando chunks: 0it [00:25, ?it/s]


ProgrammingError: 1054 (42S22): Unknown column 'FlightDate' in 'field list'